# 06 — Execution-Grounded Training Data

Generates high-quality training pairs by running generated ARO programs through the
actual ARO runtime (`aro run`) and using pass/fail as the quality signal.

Unlike notebook 03 (which uses `aro check` for syntax only), this notebook validates
**semantics**: the generated program must execute without error, and where a reference
output is available it must match.

**Strategy:**
1. **Mutation** — take an existing `Examples/` program, ask the model to write a
   variation (different data, added feature, simplified version)
2. **Recombination** — combine two examples into one application
3. **Spec-to-code** — write ARO from a natural-language description derived from proposals

**Self-repair loop:** if `aro run` fails, the error is fed back to the model
(up to `MAX_REPAIR_ATTEMPTS` times) before the pair is dropped.

**Input:**  `../../Examples/` (92 examples with `expected.txt`)
            `../data/02_knowledge/knowledge.json`
**Output:** `../data/05_exec_pairs/exec_pairs.jsonl`
            (also merged into `../data/02_knowledge/knowledge_pairs.jsonl` at end)

In [33]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json, re, sys, random, subprocess, tempfile, shutil, textwrap
from pathlib import Path
from collections import defaultdict

DATA_OUT    = GLOBAL_OUT_DIR / '../data/05_exec_pairs'
DATA_OUT.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE  = DATA_OUT / 'exec_pairs.jsonl'

# Find aro binary
ARO_BIN = shutil.which('aro') or str(ARO_ROOT / '.build/release/aro')
print(f'ARO binary: {ARO_BIN}')

with open(KB_FILE) as f:
    kb = json.load(f)

print(f'ARO root:   {ARO_ROOT}')
print(f'Examples:   {EXAMPLES_DIR}')
print(f'Knowledge:  {len(kb["actions"])} actions, {len(kb["examples"])} examples')

Config loaded | model=mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
ARO root:  /Users/kris/Projects/ARO-App
Pairs:     /Volumes/Models/MLOUT/data/02_knowledge/knowledge_pairs.jsonl
Adapters:  /Volumes/Models/MLOUT/data/adapters
ARO binary: /opt/homebrew/bin/aro
ARO root:   /Users/kris/Projects/ARO-App
Examples:   /Users/kris/Projects/ARO-App/Examples
Knowledge:  19 actions, 103 examples


## Load examples with expected output

In [34]:
def load_example(path):
    p = Path(path)
    aro_files = {}
    for f in sorted(p.rglob('*.aro')):
        aro_files[str(f.relative_to(p))] = f.read_text()
    if not aro_files:
        return None
    return {
        'name':     p.name,
        'dir':      str(p),
        'aro':      aro_files,
        'expected': (p / 'expected.txt').read_text().strip() if (p / 'expected.txt').exists() else None,
        'readme':   (p / 'README.md').read_text()            if (p / 'README.md').exists()   else None,
        'openapi':  (p / 'openapi.yaml').read_text()         if (p / 'openapi.yaml').exists() else None,
    }

# Only load examples that have expected.txt (deterministic, non-server output)
examples = []
for d in sorted(EXAMPLES_DIR.iterdir()):
    if not d.is_dir():
        continue
    ex = load_example(d)
    if ex and ex['expected'] is not None:
        examples.append(ex)

# Filter out server/keepalive examples (their expected output is timing-dependent)
def is_deterministic(ex):
    combined = ' '.join(ex['aro'].values()).lower()
    return not any(kw in combined for kw in ['keepalive', 'start the <http', 'start the <socket'])

det_examples = [e for e in examples if is_deterministic(e)]

print(f'Total examples with expected.txt: {len(examples)}')
print(f'Deterministic (runnable):         {len(det_examples)}')
print()
for e in det_examples[:5]:
    lines = len(e['aro'])
    print(f'  {e["name"]:30s}  {lines} aro file(s)  expected={len(e["expected"].splitlines())} lines')

Total examples with expected.txt: 92
Deterministic (runnable):         71

  AssertDemo                      1 aro file(s)  expected=14 lines
  AuditLogDemo                    2 aro file(s)  expected=13 lines
  AutoPipeline                    1 aro file(s)  expected=25 lines
  CSVProcessor                    1 aro file(s)  expected=14 lines
  Calculator                      2 aro file(s)  expected=22 lines


## Build system prompt

In [35]:
# Build a compact action reference for the system prompt
action_lines = []
for a in kb['actions']:
    verbs = ', '.join(a['verbs'][:3])
    preps = ', '.join(a.get('prepositions', [])[:3])
    action_lines.append(f'  {verbs:<28} prepositions: {preps}')
action_ref = '\n'.join(action_lines)

# Pull syntax summary from knowledge base
syntax_summary = kb.get('aro_syntax', '')[:3000]

SYSTEM_PROMPT = f"""You are an expert ARO (Action Result Object) programmer.
ARO is a DSL where every statement is: Verb the <Result> preposition [the] <Object>.

ARO SYNTAX RULES:
{syntax_summary}

AVAILABLE ACTIONS (verb → role):
{action_ref}

RULES:
- Every feature set: (Name: Business Activity) {{ statements }}
- Exactly one Application-Start per application
- Variables are immutable — use a new name for each transformation
- Articles (a/an/the) are optional everywhere
- String concatenation: <a> ++ <b>  (NOT +)
- For-each: For each <item> in <list> {{ ... }}
- Conditions: when <var> = value or when <expr>
- Return an <OK: status> for the <result>.

Output ONLY valid ARO code. No markdown fences unless asked. No explanations unless asked."""

print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

System prompt: 1804 chars


## Load model

In [36]:
# Auto-load warm-start adapter from notebook 04 if available
model, tokenizer, _, mlx_generate, make_sampler = load_model()

Loading mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit with warm-start adapter...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 131620.42it/s]


  Adapter: /Volumes/Models/MLOUT/data/adapters/warm_start
Model ready.


## Execution validator

Runs generated ARO code in a temp directory and checks:
1. Exit code 0
2. Output matches `expected` (if provided) — fuzzy match on key lines

In [37]:
RUN_TIMEOUT = 15   # seconds — enough for non-server examples

def write_temp_app(aro_files, extra_files=None):
    """Write aro_files dict to a temp directory, return Path."""
    tmpdir = Path(tempfile.mkdtemp(prefix='aro_exec_'))
    for rel, content in aro_files.items():
        dest = tmpdir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(content)
    if extra_files:
        for rel, content in extra_files.items():
            dest = tmpdir / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(content)
    return tmpdir

def run_aro(tmpdir):
    """Run `aro run <tmpdir>`, return (stdout, stderr, returncode)."""
    try:
        result = subprocess.run(
            [ARO_BIN, 'run', str(tmpdir)],
            capture_output=True, text=True,
            timeout=RUN_TIMEOUT,
            cwd=str(tmpdir),
        )
        return result.stdout.strip(), result.stderr.strip(), result.returncode
    except subprocess.TimeoutExpired:
        return '', 'TIMEOUT', -1
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def output_matches(actual, expected, threshold=0.6):
    """Fuzzy match: fraction of expected lines present in actual output."""
    if not expected:
        return True
    exp_lines = [l.strip() for l in expected.splitlines() if l.strip()]
    act_lower  = actual.lower()
    hits = sum(1 for l in exp_lines if l.lower() in act_lower)
    return (hits / len(exp_lines)) >= threshold

def validate_aro(aro_files, expected_output=None, extra_files=None):
    """Returns (ok, stdout, stderr)."""
    tmpdir = write_temp_app(aro_files, extra_files)
    stdout, stderr, rc = run_aro(tmpdir)
    if rc != 0:
        return False, stdout, stderr
    if expected_output and not output_matches(stdout, expected_output):
        return False, stdout, f'Output mismatch. Expected:\n{expected_output}\nGot:\n{stdout}'
    return True, stdout, stderr

print('Validator ready.')

Validator ready.


## Generation helpers

In [38]:
MAX_REPAIR_ATTEMPTS = 3
MAX_NEW_TOKENS      = 1200

def generate(prompt, max_tokens=MAX_NEW_TOKENS, temp=0.6):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    tokens = tokenizer.encode(text)
    out = mlx_generate(
        model, tokenizer,
        prompt=tokens,
        max_tokens=max_tokens,
        sampler=make_sampler(temp=temp),
        verbose=False,
    )
    # Strip any accidental markdown fences
    out = re.sub(r'^```[a-z]*\n?', '', out.strip())
    out = re.sub(r'\n?```$', '', out.strip())
    return out.strip()

def format_aro_files(aro_dict):
    """Format a multi-file example for inclusion in a prompt."""
    parts = []
    for fname, content in aro_dict.items():
        parts.append(f'## {fname}\n{content}')
    return '\n\n'.join(parts)

print('Generation helpers ready.')

Generation helpers ready.


## Generation strategies

### Strategy 1 — Mutation
Ask the model to write a variation of an existing example.

In [39]:
MUTATION_TYPES = [
    "Change the data values (use different names, numbers, or strings) but keep the same structure.",
    "Add one extra computation or log statement to demonstrate an additional operation.",
    "Simplify the example to its minimal form while keeping it runnable.",
    "Rename the variables to use a different domain (e.g. change 'user' to 'product').",
    "Add a when-guard condition that filters based on one of the computed values.",
    "Extend with a for-each loop that iterates over a small hardcoded list.",
]

def mutate_example(ex):
    """
    Returns list of (instruction, aro_files, expected_hint) tuples.
    expected_hint may be None for mutations where output is unpredictable.
    """
    mutation = random.choice(MUTATION_TYPES)
    src = format_aro_files(ex['aro'])

    instruction = (
        f"Here is an ARO program called '{ex['name']}':\n\n"
        f"{src}\n\n"
        f"Write a variation that: {mutation}\n"
        f"Output only the complete ARO source. "
        f"If multiple files are needed, separate them with '## filename.aro' headers."
    )
    return instruction, mutation


def try_generate_and_validate(instruction, reference_example=None, strategy='mutation', label=''):
    """
    Generates code, validates with aro run, self-repairs on failure.
    Returns (pair_dict, stats_str) or (None, reason).
    """
    tag = f'[{strategy}]' + (f' {label}' if label else '')
    tqdm.write(f'  {tag} generating...')
    generated = generate(instruction)

    # Parse multi-file output
    def parse_output(text):
        file_re = re.compile(r'^##\s+(\S+\.aro)', re.MULTILINE)
        parts = file_re.split(text)
        if len(parts) == 1:
            return {'main.aro': text}
        files = {}
        for i in range(1, len(parts), 2):
            fname   = parts[i].strip()
            content = parts[i+1].strip() if i+1 < len(parts) else ''
            files[fname] = content
        return files

    aro_files = parse_output(generated)
    extra = {'openapi.yaml': reference_example['openapi']} if reference_example and reference_example.get('openapi') else None

    _preview = list(aro_files.values())[0][:600].replace('\n', '\n      ')
    tqdm.write(f'  {tag} validating:\n      {_preview}')
    ok, stdout, stderr = validate_aro(aro_files, extra_files=extra)
    if ok:
        tqdm.write(f'  {tag} passed (no repairs needed)')

    repair_count = 0
    while not ok and repair_count < MAX_REPAIR_ATTEMPTS:
        repair_count += 1
        _err_parts = []
        if stderr: _err_parts.append('stderr: ' + stderr.strip()[-600:])
        if stdout: _err_parts.append('stdout (tail): ' + '\n'.join(stdout.strip().splitlines()[-15:]))
        err_msg = ('\n'.join(_err_parts) or 'unknown error').replace('\n', '\n      ')
        tqdm.write(f'  {tag} attempt {repair_count}/{MAX_REPAIR_ATTEMPTS} failed:\n      {err_msg}')
        tqdm.write(f'  {tag}   -> repairing...')
        repair_prompt = (
            f"{instruction}\n\n"
            f"Your previous attempt failed with this error:\n{stderr or stdout}\n\n"
            f"Fix the ARO code and output only the corrected source."
        )
        generated = generate(repair_prompt, temp=0.3)
        aro_files  = parse_output(generated)
        _preview = list(aro_files.values())[0][:600].replace('\n', '\n      ')
        tqdm.write(f'  {tag} validating repair {repair_count}:\n      {_preview}')
        ok, stdout, stderr = validate_aro(aro_files, extra_files=extra)
        if ok:
            tqdm.write(f'  {tag} repaired on attempt {repair_count}')

    if not ok:
        _err_parts = []
        if stderr: _err_parts.append('stderr: ' + stderr.strip()[-600:])
        if stdout: _err_parts.append('stdout (tail): ' + '\n'.join(stdout.strip().splitlines()[-15:]))
        err_msg = ('\n'.join(_err_parts) or 'unknown error').replace('\n', '\n      ')
        tqdm.write(f'  {tag} gave up after {repair_count} repairs:\n      {err_msg}')
        return None, f'failed after {repair_count} repairs: {(stderr or stdout)[:120]}'

    # Build the training pair
    code_str = format_aro_files(aro_files) if len(aro_files) > 1 else list(aro_files.values())[0]
    pair = {
        'instruction': instruction,
        'output':      code_str,
        'source':      strategy,
        'score':       1.0,
        'exec_stdout': stdout[:500],
        'repairs':     repair_count,
    }
    return pair, f'ok (repairs={repair_count}, output={len(stdout)} chars)'

print('Strategy helpers ready.')

Strategy helpers ready.


### Strategy 2 — Recombination
Combine two examples into one application.

In [40]:
def recombine_examples(ex_a, ex_b):
    src_a = format_aro_files(ex_a['aro'])
    src_b = format_aro_files(ex_b['aro'])

    instruction = (
        f"Below are two separate ARO programs.\n\n"
        f"Program A — '{ex_a['name']}':\n{src_a}\n\n"
        f"Program B — '{ex_b['name']}':\n{src_b}\n\n"
        f"Write a single ARO application that combines both programs into one. "
        f"Keep all feature sets from both. Merge them into a coherent Application-Start. "
        f"Output only the complete ARO source. Use '## filename.aro' headers if using multiple files."
    )
    return instruction

print('Recombination strategy ready.')

Recombination strategy ready.


### Strategy 3 — Spec-to-code
Generate ARO from a natural-language description extracted from proposals.

In [41]:
# Collect spec prompts from proposal Q&A seeds
spec_prompts = []
for prop in kb.get('proposals', []):
    for seed in prop.get('qa_seeds', []):
        if seed.get('code_examples'):
            # Use the heading as the natural-language spec
            spec_prompts.append({
                'heading':  seed['heading'],
                'code':     seed['code_examples'][0],
                'proposal': seed['proposal'],
            })

random.shuffle(spec_prompts)
print(f'Spec prompts from proposals: {len(spec_prompts)}')
if spec_prompts:
    print(f'  Example: "{spec_prompts[0]["heading"]}"')

def spec_to_code(spec):
    instruction = (
        f"Write a complete, runnable ARO application that demonstrates: {spec['heading']}.\n"
        f"The application must have an Application-Start feature set and "
        f"produce visible output (use Log statements).\n"
        f"Output only the ARO source code."
    )
    return instruction

print('Spec-to-code strategy ready.')

Spec prompts from proposals: 462
  Example: "3.5 Response Data"
Spec-to-code strategy ready.


## Main generation loop

In [42]:
try:
    import ipywidgets
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# --- Targets ---
TARGETS = {
    'mutation':       150,   # variations of existing examples
    'recombination':   50,   # combined applications
    'spec_to_code':   100,   # from proposal specs
}
TOTAL = sum(TARGETS.values())

results    = []
stats      = defaultdict(lambda: {'ok': 0, 'fail': 0})
already    = set()

# Resume: load already-generated pairs
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                p = json.loads(line)
                results.append(p)
                already.add(p['instruction'][:80])
    print(f'Resuming — loaded {len(results)} existing pairs')

outf = open(OUTPUT_FILE, 'a')

pbar = tqdm(total=TOTAL - len(results), desc='Exec-grounded pairs', unit='pair')

try:
    # ── Strategy 1: Mutation ────────────────────────────────────────────────
    mut_done = sum(1 for r in results if r.get('source') == 'mutation')
    for _ in range(TARGETS['mutation'] - mut_done):
        ex = random.choice(det_examples)
        instruction, mutation_desc = mutate_example(ex)
        key = instruction[:80]
        if key in already:
            continue

        tqdm.write(f'  [mutation] {ex["name"]} — {mutation_desc}')
        pair, reason = try_generate_and_validate(instruction, ex, 'mutation', label=ex['name'])
        if pair:
            results.append(pair)
            outf.write(json.dumps(pair) + '\n')
            outf.flush()
            already.add(key)
            stats['mutation']['ok'] += 1
            pbar.set_postfix({'mut_ok': stats['mutation']['ok'], 'mut_fail': stats['mutation']['fail']})
        else:
            stats['mutation']['fail'] += 1
        pbar.update(1)

    # ── Strategy 2: Recombination ────────────────────────────────────────────
    rec_done = sum(1 for r in results if r.get('source') == 'recombination')
    for _ in range(TARGETS['recombination'] - rec_done):
        ex_a, ex_b = random.sample(det_examples, 2)
        instruction = recombine_examples(ex_a, ex_b)
        key = instruction[:80]
        if key in already:
            continue

        tqdm.write(f'  [recombination] {ex_a["name"]} + {ex_b["name"]}')
        pair, reason = try_generate_and_validate(instruction, None, 'recombination', label=f'{ex_a["name"]}+{ex_b["name"]}')
        if pair:
            results.append(pair)
            outf.write(json.dumps(pair) + '\n')
            outf.flush()
            already.add(key)
            stats['recombination']['ok'] += 1
        else:
            stats['recombination']['fail'] += 1
        pbar.update(1)

    # ── Strategy 3: Spec-to-code ─────────────────────────────────────────────
    spec_done = sum(1 for r in results if r.get('source') == 'spec_to_code')
    for spec in random.sample(spec_prompts, min(TARGETS['spec_to_code'] * 2, len(spec_prompts))):
        if spec_done >= TARGETS['spec_to_code']:
            break
        instruction = spec_to_code(spec)
        key = instruction[:80]
        if key in already:
            continue

        tqdm.write(f'  [spec_to_code] {spec["heading"][:60]}')
        pair, reason = try_generate_and_validate(instruction, None, 'spec_to_code', label=spec['heading'][:50])
        if pair:
            results.append(pair)
            outf.write(json.dumps(pair) + '\n')
            outf.flush()
            already.add(key)
            stats['spec_to_code']['ok'] += 1
            spec_done += 1
        else:
            stats['spec_to_code']['fail'] += 1
        pbar.update(1)

finally:
    pbar.close()
    outf.close()

print('\n=== Results ===')
for strategy, s in stats.items():
    total_s = s['ok'] + s['fail']
    rate    = s['ok'] / total_s * 100 if total_s else 0
    print(f'  {strategy:<18} ok={s["ok"]:3d}  fail={s["fail"]:3d}  pass_rate={rate:.0f}%')
print(f'  Total saved: {len(results)} pairs → {OUTPUT_FILE}')

Resuming — loaded 12 existing pairs


Exec-grounded pairs:   0%|          | 0/288 [00:00<?, ?pair/s]

  [mutation] WhileLoop — Extend with a for-each loop that iterates over a small hardcoded list.
  [mutation] WhileLoop generating...


Exec-grounded pairs:   0%|          | 1/288 [00:21<1:42:39, 21.46s/pair, mut_ok=1, mut_fail=0]

  [mutation] WhileLoop validating:
      (* ARO-0131: While Loop Demo *)
      (* Demonstrates while loops: counter, accumulation, and early exit *)
      
      (Application-Start: Loop Demo) {
          Require the <console> from the <framework>.
      
          (* 1. Basic counter loop *)
          Log "=== Counter ===" to the <console>.
          Compute the <count> from 1.
          while <count> <= 3 {
              Log <count> to the <console>.
              Compute the <count> from <count> + 1.
          }
      
          (* 2. Accumulation loop: sum 1 + 2 + 3 + 4 + 5 = 15 *)
          Log "=== Sum ===" to the <console>.
          Compute the <i> from 1.
          Compute the <total> from 0.
          while <i> <= 5 {
              Com
  [mutation] WhileLoop passed (no repairs needed)
  [mutation] GreetingPlugin — Extend with a for-each loop that iterates over a small hardcoded list.
  [mutation] GreetingPlugin generating...


...

  [mutation] QualifierPluginPython validating:
      (Application-Start: QualifierPluginPython Demo) {
          (* Create sample lists *)
          Create the <prices> with [45, 12, 78, 33, 56].
          Log "Original prices:" to the <console>.
          Log <prices> to the <console>.
      
          (* Use the sort qualifier *)
          Compute the <sorted-prices: stats.sort> from the <prices>.
          Log "Sorted:" to the <console>.
          Log <sorted-prices> to the <console>.
      
          (* Use the min and max qualifiers *)
          Compute the <low> from the <prices>.
          Compute the <high> from the <prices>.
          Log "Lowest:" to the <console>.
          Log <low> to the <console>.
          Log "Highest:" to
  [mutation] QualifierPluginPython passed (no repairs needed)
  [mutation] DirectoryReplicatorEvents — Rename the variables to use a different domain (e.g. change 'user' to 'product').
  [mutation] DirectoryReplicatorEvents generating...


Exec-grounded pairs:   4%|▍         | 11/288 [05:17<2:29:34, 32.40s/pair, mut_ok=8, mut_fail=3]

  [mutation] DirectoryReplicatorEvents validating:
      (Produce the <output> for the <result>) {
          Create the <template-path> with "./template".
          Log "Scanning template directory..." to the <console>.
      
          (* 2. List all entries recursively *)
          List the <all-entries: recursively> from the <directory: template-path>.
      
          (* 3. Filter to get only directories *)
          Filter the <directories: List> from the <all-entries>
              where <isDirectory> is true.
      
          (* 4. Log count *)
          Compute the <count: length> from the <directories>.
          Log "Found ${count} directories" to the <console>.
      
          (* 5. Store the entire list - runtime emits obser
  [mutation] DirectoryReplicatorEvents attempt 1/3 failed:
      Compilation errors:
        error [1:10]: Expected ':', but got article(the)
        error [1:10]: Expected ':', but got article(the)
        error [20:8]: Expected ':', but got article(t

Exec-grounded pairs:   4%|▍         | 11/288 [05:22<2:15:21, 29.32s/pair, mut_ok=8, mut_fail=3]


KeyboardInterrupt: 

## Merge into main knowledge pairs

In [ ]:
# Append exec-grounded pairs to the main knowledge_pairs.jsonl used by notebook 04
existing_keys = set()
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                p = json.loads(line)
                existing_keys.add(p.get('instruction', '')[:80])

new_count = 0
with open(PAIRS_FILE, 'a') as f:
    for pair in results:
        key = pair.get('instruction', '')[:80]
        if key not in existing_keys:
            # Strip exec metadata before merging (not needed for training format)
            clean = {k: v for k, v in pair.items() if k not in ('exec_stdout', 'repairs')}
            f.write(json.dumps(clean) + '\n')
            existing_keys.add(key)
            new_count += 1

print(f'Merged {new_count} new exec-grounded pairs into {PAIRS_FILE}')
print(f'Total pairs now: {len(existing_keys)}')
print()
print('Next steps:')
print('  → Run notebook 04 again (warm-start fine-tune) to train on exec-grounded pairs')
print('  → Or proceed to notebook 06 (book Q&A) and continue the pipeline')